# 3DD-TTA — Yeni Colab oturumu kurulumu

Yeni A100 / yüksek RAM oturumlarında bu dosyadaki on kod hücresini sırayla kullanın.
**Run all kullanmayın.** İlk hücre Conda kurulumundan sonra kernel'i yeniden başlatır;
aynı VM'ye yeniden bağlanıp ikinci hücreden devam edin. CLI her çağrıda `/content`
dizinine döner; 4–10. hücreleri `/content/3DD-TTA` içinde çalıştırın.

- Hücre 2: yalnızca bilinen `fastrlock` pip hatası kullanıcı kararıyla ertelenmiştir.
  `3dd_tta_env` yoksa veya başka bir hata varsa devam etmeyin.
- Hücre 3: Drive yetkilendirmesini kullanıcı tamamlar; ardından KNN wheel kopyalanır.
- Hücre 4: PyTorch cu121/KNN/proje bağımlılıkları ile CUDA12.1/GCC11.4 araç zincirini hazırlar.
- Hücre 5–8: sırasıyla EMD, Chamfer, PointNet++ ve PVCNN+GPU smoke fazlarını çalıştırır.
  Her faz UTC heartbeat ve kalıcı completion marker üretir; başarılı faz tekrar çalıştırılırsa atlanır.
- Hücre 9: yalnızca `modelnet40_c.zip` içindeki temiz ve bozulmuş verileri hazırlar.
- Hücre 10: ağırlıkları hazırlar; native GPU, veri ve checkpoint kimliklerini doğrular.
  Rapor: `/content/cli_setup_verification.json`.

Yalnızca **1 saati aşması beklenen GPU aşamaları** için ayrıca kullanıcı onayı alın.

Doğrulama (2026-10-17): A100-SXM4-80GB / yüksek RAM üzerinde birleşik eski Cell4 ile
CUDA12.1, PyTorch2.1.2+cu121 ve GCC11.4 doğrulandı; EMD, Chamfer, PointNet++ ve PVCNN
kuruldu, GPU kontrolleri geçti. On hücreli bölünmüş sürüm yeni bir VM üzerinde henüz
baştan sona tekrar oynatılmadı. Ayrı `modelnet40_original.zip` kullanılmaz.

Kaynak kurulum davranışı tam araştırma notebook'unun ilk altı kod hücresinden gelir;
bu dosyada eski Cell4 gözlemlenebilirlik ve devam edilebilirlik için beş faza bölünmüştür.


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import condacolab
condacolab.check()

# Projeyi klonla ve dev branşına geç
!git clone https://github.com/BatuhanOrhon/3DD-TTA.git
%cd 3DD-TTA
!git checkout dev

# 1. Yazarın 3dd_tta_env (Python 3.8) ortamını env.yaml dosyasından yaratıyoruz:
!conda env create -f env.yaml

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

source_path = "/content/drive/MyDrive/thesis/KNN_CUDA-0.2-py3-none-any.whl"
dest_path = "/content/KNN_CUDA-0.2-py3-none-any.whl"

if os.path.exists(source_path):
    shutil.copy(source_path, dest_path)
    print(f"✅ Dosya başarıyla kopyalandı: {dest_path}")
else:
    print(f"❌ HATA: Kaynak dosya bulunamadı: {source_path}")

In [ ]:
# 1. Python/CUDA bağımlılıklarını ve derleme araç zincirini hazırla
!conda run -n 3dd_tta_env pip install torch==2.1.2+cu121 torchvision==0.16.2+cu121 torchaudio==2.1.2+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# 2. KNN_CUDA kurulumu (/content altındaki doğrulanmış wheel)
!conda run -n 3dd_tta_env pip install /content/KNN_CUDA-0.2-py3-none-any.whl

# 3. Proje bağımlılıkları ve CLIP
!conda run -n 3dd_tta_env pip install -r requirements.txt
!conda run -n 3dd_tta_env pip install git+https://github.com/openai/CLIP.git

import subprocess
from pathlib import Path

def _tta_run_checked(args):
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, args)

_tta_run_checked([
    "conda", "install", "-y", "-n", "3dd_tta_env", "--override-channels",
    "-c", "nvidia/label/cuda-12.1.1", "-c", "conda-forge",
    "cuda-toolkit=12.1.1", "gcc_linux-64=11.4.0", "gxx_linux-64=11.4.0",
])

# Bağımsız kopya: sonraki hücreler yalnızca bir fazı çalıştırır.
_tta_build_source = r'''#!/usr/bin/env bash
set -euo pipefail

TTA_PHASE="${1:-all}"
case "$TTA_PHASE" in
  prepare|emd|chamfer|pointnet2|pvcnn-smoke|all) ;;
  *)
    echo "Unknown build phase: $TTA_PHASE" >&2
    exit 2
    ;;
esac

TTA_REPO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)"
TTA_STATE_DIR="$TTA_REPO_DIR/.colab_setup/native"
mkdir -p "$TTA_STATE_DIR"
TTA_ENV_PREFIX="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import sys; print(sys.prefix)')"
TTA_TORCH_LIB="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import os, torch; print(os.path.join(os.path.dirname(torch.__file__), "lib"))')"
TTA_HOST_CC="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import os; print(os.environ["CC"])')"
TTA_HOST_CXX="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import os; print(os.environ["CXX"])')"
TTA_GPU_ARCH="$(conda run --no-capture-output -n 3dd_tta_env python -c 'import torch; assert torch.version.cuda == "12.1", torch.version.cuda; assert torch.cuda.is_available(), "Select a Colab GPU runtime"; print(".".join(map(str, torch.cuda.get_device_capability())))')"
TTA_CUDA_INCLUDE="$TTA_ENV_PREFIX/targets/x86_64-linux/include"
TTA_CUDA_LIB="$TTA_ENV_PREFIX/targets/x86_64-linux/lib"

run_with_cuda_env() {
  conda run --no-capture-output -n 3dd_tta_env env \
    CC="$TTA_HOST_CC" CXX="$TTA_HOST_CXX" \
    CUDA_HOME="$TTA_ENV_PREFIX" \
    CPATH="$TTA_CUDA_INCLUDE${CPATH:+:$CPATH}" \
    LIBRARY_PATH="$TTA_CUDA_LIB${LIBRARY_PATH:+:$LIBRARY_PATH}" \
    LD_LIBRARY_PATH="$TTA_TORCH_LIB:$TTA_CUDA_LIB${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}" \
    TORCH_CUDA_ARCH_LIST="$TTA_GPU_ARCH" MAX_JOBS="${MAX_JOBS:-1}" "$@"
}

heartbeat() {
  printf '%s phase=%s %s\n' "$(date -u +%Y-%m-%dT%H:%M:%SZ)" "$1" "$2"
}

run_with_heartbeat() {
  local phase="$1"
  shift
  heartbeat "$phase" "started"
  "$@" &
  local child=$!
  while kill -0 "$child" 2>/dev/null; do
    local tick
    for tick in {1..12}; do
      sleep 10
      kill -0 "$child" 2>/dev/null || break
    done
    kill -0 "$child" 2>/dev/null && heartbeat "$phase" "still running"
  done
  if wait "$child"; then
    heartbeat "$phase" "completed"
  else
    local status=$?
    heartbeat "$phase" "failed status=$status"
    return "$status"
  fi
}

require_phase() {
  test -f "$TTA_STATE_DIR/$1.complete" || {
    echo "Required build phase is incomplete: $1" >&2
    exit 1
  }
}

mark_phase() {
  touch "$TTA_STATE_DIR/$1.complete"
}

prepare_sources() {
  if [[ -f "$TTA_STATE_DIR/prepare.complete" ]]; then
    heartbeat prepare "already complete; skipping"
    return
  fi
  heartbeat prepare "validating CUDA 12.1 toolchain"
  # NVIDIA toolkit releases use either targets/ or the environment root layout.
  if [[ ! -f "$TTA_CUDA_INCLUDE/cuda_runtime_api.h" ]]; then
    TTA_CUDA_INCLUDE="$TTA_ENV_PREFIX/include"
  fi
  if [[ ! -e "$TTA_CUDA_LIB/libcudart.so" ]]; then
    TTA_CUDA_LIB="$TTA_ENV_PREFIX/lib"
  fi
  test -x "$TTA_ENV_PREFIX/bin/nvcc"
  test -f "$TTA_CUDA_INCLUDE/cuda_runtime_api.h"
  test -e "$TTA_CUDA_LIB/libcudart.so"
  conda run --no-capture-output -n 3dd_tta_env python - "$TTA_ENV_PREFIX/bin/nvcc" <<'PY'
import subprocess, sys
version = subprocess.check_output([sys.argv[1], '--version'], text=True)
print(version)
assert 'release 12.1,' in version, 'CUDA compiler must match PyTorch cu121'
PY
  echo "GPU architecture: $TTA_GPU_ARCH"
  # Future conda-run/JIT calls must use the same toolkit as this build.
  conda env config vars set -n 3dd_tta_env \
    CUDA_HOME="$TTA_ENV_PREFIX" \
    CPATH="$TTA_CUDA_INCLUDE${CPATH:+:$CPATH}" \
    LIBRARY_PATH="$TTA_CUDA_LIB${LIBRARY_PATH:+:$LIBRARY_PATH}" \
    LD_LIBRARY_PATH="$TTA_TORCH_LIB:$TTA_CUDA_LIB${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}" \
    TORCH_CUDA_ARCH_LIST="$TTA_GPU_ARCH"

  # The notebook checks out dev, whose setup.py overwrites the selected GPU target.
  conda run --no-capture-output -n 3dd_tta_env python - "$TTA_REPO_DIR" <<'PY'
import hashlib, sys
from pathlib import Path
path = Path(sys.argv[1]) / 'Pointnet2_PyTorch/pointnet2_ops_lib/setup.py'
source = path.read_text()
old = 'os.environ["TORCH_CUDA_ARCH_LIST"] = "7.0;7.5;8.0;8.6"'
updated = source.replace(old, 'os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "8.6")')
if updated != source:
    print('PointNet++ setup SHA256:', hashlib.sha256(source.encode()).hexdigest(),
          '->', hashlib.sha256(updated.encode()).hexdigest())
    path.write_text(updated)

# The extension copy lacks cost kernels; reuse the project's complete EMD source.
kernel = Path(sys.argv[1]) / 'extensions/emd/cuda/emd_kernel.cu'
complete = Path(sys.argv[1]) / 'third_party/PyTorchEMD/cuda/emd_kernel.cu'
required = ('at::Tensor MatchCostForward(', 'std::vector<at::Tensor> MatchCostBackward(')
source = kernel.read_text()
if not all(symbol in source for symbol in required):
    replacement = complete.read_text()
    assert all(symbol in replacement for symbol in required), 'Complete EMD kernels missing'
    print('EMD kernel SHA256:', hashlib.sha256(source.encode()).hexdigest(),
          '->', hashlib.sha256(replacement.encode()).hexdigest())
    kernel.write_text(replacement)
PY
  mark_phase prepare
  heartbeat prepare "completed"
}

build_extension() {
  local phase="$1"
  local relative_dir="$2"
  require_phase prepare
  if [[ -f "$TTA_STATE_DIR/$phase.complete" ]]; then
    heartbeat "$phase" "already complete; skipping"
    return
  fi
  cd "$TTA_REPO_DIR/$relative_dir"
  # Objects built with the system toolkit must not enter the cu121 rebuild.
  rm -rf -- "$TTA_REPO_DIR/$relative_dir/build"
  run_with_heartbeat "$phase" run_with_cuda_env python setup.py install --force
  mark_phase "$phase"
}

build_pvcnn_and_smoke() {
  require_phase emd
  require_phase chamfer
  require_phase pointnet2
  if [[ -f "$TTA_STATE_DIR/pvcnn-smoke.complete" ]]; then
    heartbeat pvcnn-smoke "already complete; skipping"
    return
  fi
  cd "$TTA_REPO_DIR"
  run_with_heartbeat pvcnn run_with_cuda_env python build_pkg.py
  heartbeat smoke "native GPU checks started"
  run_with_cuda_env python - <<'PY'
import torch, chamfer, emd_cuda, pointnet2_ops._ext
from pointnet2_ops import pointnet2_utils
points = (torch.rand(1, 32, 3, device='cuda') + 1).contiguous()
indices = pointnet2_utils.furthest_point_sample(points, 8)
assert tuple(indices.shape) == (1, 8)
assert bool(((indices >= 0) & (indices < 32)).all())
features = points.transpose(1, 2).contiguous().requires_grad_(True)
sampled = pointnet2_utils.gather_operation(features, indices)
assert tuple(sampled.shape) == (1, 3, 8)
assert torch.allclose(sampled.transpose(1, 2), points[:, indices[0].long(), :])
sampled.sum().backward()
assert bool(torch.isfinite(features.grad).all())
torch.cuda.synchronize()
print('CUDA extension checks passed:', torch.__version__, torch.version.cuda,
      torch.cuda.get_device_name(0), 'FPS + gather forward/backward')
PY
  mark_phase pvcnn-smoke
  heartbeat pvcnn-smoke "completed"
}

case "$TTA_PHASE" in
  prepare) prepare_sources ;;
  emd) build_extension emd extensions/emd ;;
  chamfer) build_extension chamfer extensions/chamfer_dist ;;
  pointnet2) build_extension pointnet2 Pointnet2_PyTorch/pointnet2_ops_lib ;;
  pvcnn-smoke) build_pvcnn_and_smoke ;;
  all)
    prepare_sources
    build_extension emd extensions/emd
    build_extension chamfer extensions/chamfer_dist
    build_extension pointnet2 Pointnet2_PyTorch/pointnet2_ops_lib
    build_pvcnn_and_smoke
    ;;
esac
'''
_tta_build_path = Path("/content/3DD-TTA/scripts/build_cuda_extensions_colab.sh")
_tta_build_path.parent.mkdir(parents=True, exist_ok=True)
_tta_build_path.write_text(_tta_build_source)
_tta_run_checked(["bash", str(_tta_build_path), "prepare"])
print("Cell4a tamamlandı: CUDA 12.1/GCC11.4 doğrulandı ve kaynaklar hazırlandı.", flush=True)


In [ ]:
from datetime import datetime, timezone
import subprocess

print(datetime.now(timezone.utc).isoformat(), "Cell4b/EMD başlıyor", flush=True)
subprocess.run(["bash", "/content/3DD-TTA/scripts/build_cuda_extensions_colab.sh", "emd"], check=True)
print(datetime.now(timezone.utc).isoformat(), "Cell4b/EMD tamamlandı", flush=True)


In [ ]:
from datetime import datetime, timezone
import subprocess

print(datetime.now(timezone.utc).isoformat(), "Cell4c/Chamfer başlıyor", flush=True)
subprocess.run(["bash", "/content/3DD-TTA/scripts/build_cuda_extensions_colab.sh", "chamfer"], check=True)
print(datetime.now(timezone.utc).isoformat(), "Cell4c/Chamfer tamamlandı", flush=True)


In [ ]:
from datetime import datetime, timezone
import subprocess

print(datetime.now(timezone.utc).isoformat(), "Cell4d/PointNet++ başlıyor", flush=True)
subprocess.run(["bash", "/content/3DD-TTA/scripts/build_cuda_extensions_colab.sh", "pointnet2"], check=True)
print(datetime.now(timezone.utc).isoformat(), "Cell4d/PointNet++ tamamlandı", flush=True)


In [ ]:
from datetime import datetime, timezone
import subprocess

print(datetime.now(timezone.utc).isoformat(), "Cell4e/PVCNN+GPU smoke başlıyor", flush=True)
subprocess.run(["bash", "/content/3DD-TTA/scripts/build_cuda_extensions_colab.sh", "pvcnn-smoke"], check=True)
print(datetime.now(timezone.utc).isoformat(), "Cell4e/PVCNN+GPU smoke tamamlandı", flush=True)


In [ ]:
from pathlib import Path
import shutil, urllib.request, zipfile

# ModelNet40-C archive includes the clean data_original.npy and corruptions.
drive_dataset_dir = Path('/content/drive/MyDrive/thesis/dataset')
local_data_dir = Path('/content/3DD-TTA/data')
local_data_dir.mkdir(parents=True, exist_ok=True)
corrupted_zip_drive = drive_dataset_dir / 'modelnet40_c.zip'
corrupted_zip_local = local_data_dir / 'modelnet40_c.zip'
if corrupted_zip_drive.is_file():
    assert zipfile.is_zipfile(corrupted_zip_drive), 'Drive modelnet40_c.zip is not a valid ZIP'
    print('ModelNet40-C (clean + corruptions) Drive arşivinden kopyalanıyor...', flush=True)
    shutil.copyfile(corrupted_zip_drive, corrupted_zip_local)
else:
    print('ModelNet40-C Zenodo kaynağından indiriliyor...', flush=True)
    partial = corrupted_zip_local.with_suffix('.zip.part')
    urllib.request.urlretrieve('https://zenodo.org/records/6017834/files/modelnet40_c.zip', partial)
    assert zipfile.is_zipfile(partial), 'Downloaded ModelNet40-C is not a valid ZIP'
    partial.replace(corrupted_zip_local)

with zipfile.ZipFile(corrupted_zip_local) as archive:
    root = local_data_dir.resolve()
    members = [item for item in archive.infolist() if item.filename.startswith("modelnet40_c/")]
    assert members, "ModelNet40-C archive has no dataset folder"
    for member in members:
        destination = (root / member.filename).resolve()
        assert root in destination.parents, 'Unsafe ModelNet40-C archive path'
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'Archive symlinks are not allowed'
    assert archive.testzip() is None, 'ModelNet40-C ZIP CRC check failed'
    archive.extractall(local_data_dir, members=members)
assert (local_data_dir / 'modelnet40_c/data_original.npy').is_file(), 'Clean ModelNet40-C array is missing'
if not corrupted_zip_drive.exists():
    drive_dataset_dir.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(corrupted_zip_local, corrupted_zip_drive)
print('ModelNet40-C temiz ve bozulmuş verileri hazır.', flush=True)



In [ ]:
import os
import shutil
!pip install -q gdown
local_pointnet_dir = "/content/3DD-TTA/pointnet_ckpts"
local_lion_dir = "/content/3DD-TTA/lion_ckpts"
os.makedirs(local_pointnet_dir, exist_ok=True)
os.makedirs(local_lion_dir, exist_ok=True)
drive_weights_dir = "/content/drive/MyDrive/thesis/weights"
os.makedirs(drive_weights_dir, exist_ok=True)
# 1. PointMAE Weights
pointmae_drive = os.path.join(drive_weights_dir, "modelnet_jt.pth")
pointmae_local = os.path.join(local_pointnet_dir, "modelnet_jt.pth")
if os.path.exists(pointmae_drive):
    print("PointMAE ağırlıkları Drive'dan kopyalanıyor...")
    shutil.copy(pointmae_drive, pointmae_local)
else:
    print("PointMAE ağırlıkları indiriliyor...")
    !gdown --folder https://drive.google.com/drive/folders/1MTH8WpOqfAIiZ0DZV9p-tSKiDgQ_Id5A?usp=sharing -O /content/temp_pointnet/
    !find /content/temp_pointnet -name "modelnet_jt.pth" -exec mv {} {local_pointnet_dir}/ \;
    !rm -rf /content/temp_pointnet
    print("PointMAE ağırlıkları Drive'a yedekleniyor...")
    if os.path.exists(pointmae_local):
        shutil.copy(pointmae_local, pointmae_drive)
# 2. LION Diffusion Weights
lion_drive = os.path.join(drive_weights_dir, "epoch_10999_iters_2100999.pt")
lion_local = os.path.join(local_lion_dir, "epoch_10999_iters_2100999.pt")
if os.path.exists(lion_drive):
    print("LION ağırlıkları Drive'dan kopyalanıyor...")
    shutil.copy(lion_drive, lion_local)
else:
    print("LION ağırlıkları indiriliyor...")
    !wget -O {lion_local} -nc https://huggingface.co/xiaohui2022/lion_ckpt/resolve/main/unconditional/all55/checkpoints/epoch_10999_iters_2100999.pt
    print("LION ağırlıkları Drive'a yedekleniyor...")
    if os.path.exists(lion_local):
        shutil.copy(lion_local, lion_drive)
print("✅ Model ağırlıkları (LION ve PointMAE) kullanıma hazır!")


# Son kontrol: native GPU işlemleri, ModelNet40-C ve checkpoint kimlikleri.
from pathlib import Path
import subprocess
_tta_verify_source = "import os, sys, json, hashlib, subprocess, importlib\nfrom pathlib import Path\nos.chdir('/content/3DD-TTA')\nsys.path.insert(0,os.getcwd())\nassert sys.version_info[:2] == (3, 8), sys.version\nimport torch, numpy as np\nassert torch.cuda.is_available()\nassert 'A100' in torch.cuda.get_device_name(0)\na=torch.arange(32,device='cuda',dtype=torch.float32)\nassert float(a.sum().cpu())==496\nmodules=['knn_cuda','emd_cuda','chamfer','pointnet2_ops._ext','clip','open3d','h5py','diffusers','timm','models.pvcnn2','utils.eval_helper']\nimports={name:str(importlib.import_module(name).__file__) for name in modules}\nfrom pointnet2_ops import pointnet2_utils\nfrom knn_cuda import KNN\nfrom extensions.chamfer_dist import ChamferDistanceL2\nfrom extensions.emd import emd\nx=torch.rand(1,32,3,device='cuda').contiguous()\ni=pointnet2_utils.furthest_point_sample(x,8)\nassert tuple(i.shape)==(1,8) and bool(((i>=0)&(i<32)).all())\nd,idx=KNN(k=4,transpose_mode=True)(x,x)\nassert bool(torch.isfinite(d).all()) and idx.numel()==128\ncd=ChamferDistanceL2()(x,x)\nassert bool(torch.isfinite(cd).all()) and abs(float(cd))<1e-5\ncost=emd()(x,x)\nassert bool(torch.isfinite(cost).all())\ndef identity(p):\n p=Path(p);assert p.is_file() and p.stat().st_size>0,str(p)\n h=hashlib.sha256()\n with p.open('rb') as f:\n  for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)\n return {'bytes':p.stat().st_size,'sha256':h.hexdigest()}\nassets={s:identity(s) for s in ['env.yaml','requirements.txt','scripts/build_cuda_extensions_colab.sh','/content/KNN_CUDA-0.2-py3-none-any.whl','pointnet_ckpts/modelnet_jt.pth','lion_ckpts/epoch_10999_iters_2100999.pt']}\nroot=Path('data/modelnet40_c')\nassert root.is_dir()\nclean=np.load(str(root/'data_original.npy'),mmap_mode='r')\nassert clean.ndim==3 and clean.shape[0]==2468 and clean.shape[2]==3,clean.shape\nassets[str(root/'data_original.npy')]=identity(root/'data_original.npy')\nassets['data/modelnet40_c.zip']=identity('data/modelnet40_c.zip')\narrays=sorted(root.glob('data_*_5.npy'))\nassert len(arrays)>=15,len(arrays)\nassert (root/'label.npy').is_file()\nassets[str(root/'label.npy')]=identity(root/'label.npy')\nfor f in arrays:\n assets[str(f)]=identity(f)\n z=np.load(str(f),mmap_mode='r');assert z.ndim==3 and z.shape[0]==2468 and z.shape[2]==3,(str(f),z.shape)\nassert Path('/content/drive/MyDrive/thesis/dataset').is_dir()\ninfo={'status':'passed','python':sys.version,'torch':torch.__version__,'torch_cuda':torch.version.cuda,'numpy':np.__version__,'gpu':torch.cuda.get_device_name(0),'gpu_memory_bytes':torch.cuda.get_device_properties(0).total_memory,'meminfo':Path('/proc/meminfo').read_text().splitlines()[:3],'remote_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'remote_branch':subprocess.check_output(['git','branch','--show-current'],text=True).strip(),'remote_status':subprocess.check_output(['git','status','--short'],text=True),'imports':imports,'native_checks':['CUDA tensor sum','FPS','KNN','Chamfer identical input','EMD finite output'],'assets':assets,'modelnet_corruption_arrays':len(arrays)}\nPath('/content/cli_setup_verification.json').write_text(json.dumps(info,indent=2)+'\\n')\nprint(json.dumps(info,indent=2));print('SETUP_ENVIRONMENT_VERIFIED')\n"
_tta_verify_path = Path('/content/3DD-TTA/scripts/verify_colab_environment.py')
_tta_verify_path.parent.mkdir(parents=True, exist_ok=True)
_tta_verify_path.write_text(_tta_verify_source)
_tta_verify_command = ['conda', 'run', '--no-capture-output', '-n', '3dd_tta_env',
                       'python', str(_tta_verify_path)]
with subprocess.Popen(_tta_verify_command, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as _tta_process:
    for _tta_line in _tta_process.stdout:
        print(_tta_line, end='', flush=True)
    if _tta_process.wait():
        raise subprocess.CalledProcessError(_tta_process.returncode, _tta_verify_command)
